<a href="https://colab.research.google.com/github/prostudentme-lang/AUTOMATIC-TOLL-SYSTEM-AND-VEHICLE-IDENTIFICATION-ALONG-WITH-CAR-BRANDS-USING-CV/blob/main/AutomaticTollSystemAndVehicleIdentificationUsingCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bing-image-downloader

from bing_image_downloader import downloader

# Download images for each class
downloader.download("car tyre", limit=50, output_dir="dataset_type", adult_filter_off=True)
downloader.download("truck tyre", limit=50, output_dir="dataset_type", adult_filter_off=True)
downloader.download("auto rickshaw tyre", limit=50, output_dir="dataset_type", adult_filter_off=True)

[%] Downloading Images to /content/dataset_type/car tyre


[!!]Indexing page: 1

[%] Indexed 47 Images on Page 1.


[%] Downloading Image #1 from https://bonwaytyre.com/wp-content/uploads/2022/10/car-tire-technology-control-picture-BONWAY.jpg
[!] Issue getting: https://bonwaytyre.com/wp-content/uploads/2022/10/car-tire-technology-control-picture-BONWAY.jpg
[!] Error:: HTTP Error 503: Service Temporarily Unavailable
[%] Downloading Image #1 from https://www.kwik-fit.com/assets/images/wintertyres_howdowintertyreswork.jpg
[%] File Downloaded !

[%] Downloading Image #2 from http://sc01.alicdn.com/kf/UT8brhNXuBaXXagOFbXH/222476242/UT8brhNXuBaXXagOFbXH.jpg
[%] File Downloaded !

[%] Downloading Image #3 from https://media.licdn.com/dms/image/v2/D4E10AQHh0nZcvFk2lg/image-shrink_1280/image-shrink_1280/0/1691949610077?e=2147483647&amp;v=beta&amp;t=y4c-BQut5I8mVyw4lo8QijpFVQPLmP83i0Pqg7BhetM
[!] Issue getting: https://media.licdn.com/dms/image/v2/D4E10AQHh0nZcvFk2lg/image-shrink_1280/image-shri

In [ ]:
import zipfile
import os

zip_path = "/content/dataset_brand.zip"   # change name if needed
extract_path = "/content/dataset_brand"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")

Unzipped successfully!


In [ ]:
# ================= INSTALL =================
!apt-get install -y tesseract-ocr
!pip install pytesseract pillow

# ================= IMPORTS =================
import os
from PIL import Image
import cv2
import pytesseract
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
import numpy as np
from google.colab import files
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
import re
import random
from datetime import datetime

IMG_SIZE = 224
BATCH_SIZE = 32

# ================= CLEAN DATASET =================
valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def clean_dataset(directory):
    print(f"🔍 Cleaning {directory}...")
    for root, dirs, files in os.walk(directory):
        for file in files:
            path = os.path.join(root, file)

            # Remove unsupported extensions
            if not file.lower().endswith(valid_ext):
                print("❌ Removing unsupported:", path)
                os.remove(path)
                continue

            # Remove corrupted images
            try:
                img = Image.open(path)
                img.verify()
            except:
                print("❌ Removing corrupted:", path)
                os.remove(path)

clean_dataset("/content/dataset_type")
clean_dataset("/content/dataset_brand")





from PIL import Image
import numpy as np
import os

def load_images_safe(directory):
    images = []
    labels = []
    class_names = sorted(os.listdir(directory))

    for label, class_name in enumerate(class_names):
        class_path = os.path.join(directory, class_name)

        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            img_path = os.path.join(class_path, file)

            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize((224, 224))
                img_array = np.array(img) / 255.0

                images.append(img_array)
                labels.append(label)

            except:
                print("❌ Skipping bad file:", img_path)

    return np.array(images), np.array(labels), class_names
X_type, y_type, type_classes = load_images_safe("/content/dataset_type")
X_brand, y_brand, brand_classes = load_images_safe("/content/dataset_brand")

# ================= MODEL FUNCTION =================
def create_model(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# ================= TRAIN MODELS =================
type_model = create_model(len(type_classes))
type_model.fit(X_type, y_type, epochs=5, batch_size=32)
type_model.save("vehicle_type_model.h5")

brand_model = create_model(len(brand_classes))
brand_model.fit(X_brand, y_brand, epochs=5, batch_size=32)
brand_model.save("brand_model.h5")



In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
# ================= TRAIN TOLL MODEL =================
data = {
    "vehicle_type": ["Car", "Truck", "Auto"],
    "distance": [30, 30, 30],
    "toll_fee": [80, 100, 150]
}

df = pd.DataFrame(data)

encoder = LabelEncoder()
df["vehicle_type_enc"] = encoder.fit_transform(df["vehicle_type"])

X = df[["vehicle_type_enc", "distance"]]
y = df["toll_fee"]

toll_model = LinearRegression()
toll_model.fit(X, y)

print("✅ All models trained successfully")
print()
print()
# ================= LOAD MODELS =================
type_model = tf.keras.models.load_model("vehicle_type_model.h5",compile=False)
brand_model = tf.keras.models.load_model("brand_model.h5",compile=False)
# ================= UPLOAD IMAGE =================
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
print("📂 Uploaded:", img_path)

# ================= OCR =================
img = cv2.imread(img_path)

if img is None:
    raise ValueError("❌ Invalid image uploaded!")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

text = pytesseract.image_to_string(gray, config='--psm 8')
vehicle_number = re.sub(r'[^A-Za-z0-9]', '', text)
print()
print()
print()
print("----------------------TOLL RECIEPT----------------------")
print()
print("          Vehicle Number : ", vehicle_number if vehicle_number else "Not detected")
print()



# ================= PREDICTION =================
def predict_vehicle(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
    except:
        raise ValueError("❌ Error loading image")

    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    type_pred = type_model.predict(img_array,verbose=0)
    vehicle_type = type_classes[np.argmax(type_pred)]
     # ================= ENTRY =================
    entry_time = datetime.now()

    print("          Entry Time     : ", entry_time)
    print()
    print("          Vehicle Type   : ", vehicle_type)
    print()

    # ================= EXIT + TOLL =================
    distance = random.randint(20, 100)

    try:
        vehicle_type_enc = encoder.transform([vehicle_type.capitalize()])[0]
    except:
        print("⚠️ Unknown vehicle type, defaulting to Car")
        vehicle_type_enc = encoder.transform(["Car"])[0]

    predicted_fee = toll_model.predict([[vehicle_type_enc,distance]])[0]

    exit_time = datetime.now()

    if vehicle_type.lower() == "car":
        brand_pred = brand_model.predict(img_array,verbose=0)
        brand = brand_classes[np.argmax(brand_pred)]
        print("          Car Brand    : ", brand)
        print()
    else:
        print("          Vehicle Brand  : Heavy Vehicle")
        print()

    return vehicle_type

vehicle_type = predict_vehicle(img_path)


print("          Distance       : ", distance, "km")
print()
print("          Toll Fee       : ₹", round(predicted_fee, 2))
print()
print("          Exit Time      : ", exit_time)
print()
print("---------------------------END---------------------------")
print()
print()
print()
print()
print()



✅ All models trained successfully




Saving 7364.jpg to 7364.jpg
📂 Uploaded: 7364.jpg



----------------------TOLL RECIEPT----------------------

          Vehicle Number :  Not detected

          Entry Time     :  2026-03-31 17:20:13.333941

          Vehicle Type   :  car

          Car Brand    :  Honda_city

          Distance       :  48 km

          Toll Fee       : ₹ 137.0

          Exit Time      :  2026-03-31 15:48:17.638664

---------------------------END---------------------------





